In [1]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
from openpyxl.styles import PatternFill

# ---------------- CONFIG ---------------- #
BASE_DIR = Path.home() / "Documents/Personal Files/Finance"
INPUT_FILE = BASE_DIR / "stocks_input_Nihal.xlsx"
OUTPUT_FILE = BASE_DIR / "stocks_output_Nihal.xlsx"

# Color mapping
GREEN_FILL = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")  # No risk
ORANGE_FILL = PatternFill(start_color="FFC000", end_color="FFC000", fill_type="solid")  # One risk
RED_FILL = PatternFill(start_color="FF6666", end_color="FF6666", fill_type="solid")     # Two risks

# ---------------- HELPERS ---------------- #
def make_excel_safe_datetime(df):
    """Convert datetime columns to Excel-safe, tz-naive datetime."""
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].dt.tz_localize(None)
        elif df[col].dtype == object:
            df[col] = df[col].apply(
                lambda x: pd.Timestamp(x).replace(tzinfo=None) if isinstance(x, (datetime, pd.Timestamp)) else x
            )
    return df

def get_upcoming_events(stock, expiry_date):
    """Check for dividend/earnings risk in next 2 weeks."""
    has_dividend, has_earnings, comments = False, False, []

    try:
        ex_div = stock.info.get("exDividendDate")
        if ex_div:
            ex_div_date = datetime.fromtimestamp(ex_div).date()
            if expiry_date - timedelta(days=14) <= ex_div_date <= expiry_date:
                has_dividend = True
                comments.append("Ex-dividend risk")
    except:
        pass

    try:
        cal = stock.calendar
        if not cal.empty and "Earnings Date" in cal.index:
            earnings_date = cal.loc["Earnings Date"][0].date()
            if expiry_date - timedelta(days=14) <= earnings_date <= expiry_date:
                has_earnings = True
                comments.append("Earnings announcement")
    except:
        pass

    return has_dividend, has_earnings, "; ".join(comments)

# ---------------- DIVIDENDS ---------------- #
def get_stock_info(stock, ticker, quantity, institution):
    try:
        info = stock.info
        price = info.get("currentPrice")
        annual_dividend = info.get("dividendRate") or 0
        dividend_date = None
        if info.get("dividendDate"):
            dividend_date = datetime.fromtimestamp(info["dividendDate"])

        if annual_dividend <= 0 or not price:
            return None

        quarterly_dividend = round(annual_dividend / 4, 4)

        return {
            "Institution": institution,
            "Ticker": ticker,
            "Quantity": quantity,
            "Price": price,
            "Quarterly Dividend / Share": quarterly_dividend,
            "Quarterly Dividend Income": round(quarterly_dividend * quantity, 2),
            "Dividend Yield": info.get("dividendYield"),
            "Dividend Date": dividend_date
        }
    except:
        return None

# ---------------- COVERED CALLS ---------------- #
def get_next_week_calls(stock, ticker, quantity, institution):
    if ticker.upper() == "ORCL":
        return None

    try:
        expirations = stock.options
    except:
        return None

    today = datetime.today().date()
    expiry = next(
        (e for e in expirations
         if today <= datetime.strptime(e, "%Y-%m-%d").date() <= today + timedelta(days=7)),
        None
    )
    if not expiry:
        return None

    try:
        calls = stock.option_chain(expiry).calls
    except:
        return None

    price = stock.info.get("currentPrice")
    contracts = quantity // 100
    if not price or contracts == 0:
        return None

    calls = calls[
        (calls["strike"] > price) &
        (calls["volume"] > 10) &
        (calls["openInterest"] > 50) &
        (calls["lastPrice"] > 0)
    ]

    if calls.empty:
        return None

    expiry_date = datetime.strptime(expiry, "%Y-%m-%d").date()
    days = (expiry_date - today).days
    if days <= 0:
        return None

    has_div, has_earn, comment = get_upcoming_events(stock, expiry_date)

    calls = calls.copy()
    calls["Institution"] = institution
    calls["Ticker"] = ticker
    calls["Expiration"] = expiry
    calls["Contracts"] = contracts
    calls["Premium / Share"] = calls["lastPrice"]
    calls["Total Premium"] = calls["lastPrice"] * 100 * contracts
    calls["Return %"] = (calls["lastPrice"] / price) * 100
    calls["Annualized %"] = (calls["Return %"] / days) * 365
    calls["Dividend Risk"] = has_div
    calls["Earnings Risk"] = has_earn
    calls["Comments"] = comment
    calls["Include"] = True

    calls = calls.sort_values("Return %", ascending=False).head(5)
    calls["Group"] = range(1, len(calls) + 1)

    numeric_cols = calls.select_dtypes(include="number").columns
    calls[numeric_cols] = calls[numeric_cols].round(2)
    return calls

# ---------------- MAIN ---------------- #
def main():
    input_df = pd.read_excel(INPUT_FILE, sheet_name="Input")
    dividend_rows, call_rows = [], []

    for _, row in input_df.iterrows():
        ticker = row["Ticker"]
        qty = int(row["Quantity"])
        institution = row.get("Institution", "Unknown")
        stock = yf.Ticker(ticker)

        div = get_stock_info(stock, ticker, qty, institution)
        if div:
            dividend_rows.append(div)

        if qty >= 100:
            calls = get_next_week_calls(stock, ticker, qty, institution)
            if calls is not None:
                call_rows.append(calls)

    # ---- Dividends ---- #
    dividend_df = pd.DataFrame(dividend_rows)
    dividend_df = make_excel_safe_datetime(dividend_df)

    # Dividend pivot by month and institution
    dividend_pivot_df = dividend_df.copy()
    dividend_pivot_df["Month"] = dividend_pivot_df["Dividend Date"].dt.to_period("M").astype(str)
    dividend_pivot_df["Month"] = dividend_pivot_df["Month"].fillna("Unknown")
    dividend_monthly = dividend_pivot_df.groupby(["Institution", "Ticker", "Month"])["Quarterly Dividend Income"].sum().reset_index()
    dividend_monthly.rename(columns={"Quarterly Dividend Income": "Total Dividend Income"}, inplace=True)
    dividend_monthly = make_excel_safe_datetime(dividend_monthly)

    # Add totals per Institution
    dividend_totals = dividend_monthly.groupby("Institution")["Total Dividend Income"].sum().reset_index()
    dividend_monthly = pd.merge(dividend_monthly, dividend_totals.rename(columns={"Total Dividend Income": "Institution Total"}), on="Institution", how="left")

    # ---- Covered Calls ---- #
    if call_rows:
        calls_df = pd.concat(call_rows, ignore_index=True)
        calls_df = make_excel_safe_datetime(calls_df)
        trade_df = calls_df[calls_df["Include"] == True]
    else:
        calls_df = pd.DataFrame()
        trade_df = pd.DataFrame()

    # Covered Calls Pivot grouped by Institution/Ticker/Strike
    pivot_df = pd.DataFrame()
    if not trade_df.empty:
        pivot_df = pd.pivot_table(
            trade_df,
            values="Total Premium",
            index=["Institution", "Ticker", "strike"],
            columns=["Group"],
            aggfunc="sum",
            fill_value=0
        )
        pivot_df = make_excel_safe_datetime(pivot_df)

        # Totals per institution, inserted directly beneath that institution's own rows
        if isinstance(pivot_df.index, pd.MultiIndex) and "Institution" in pivot_df.index.names:
            blocks = []
            for inst in pivot_df.index.get_level_values("Institution").unique():
                inst_block = pivot_df[pivot_df.index.get_level_values("Institution") == inst]
                total_row = inst_block.sum()
                total_row.name = tuple(
                    inst if name == "Institution" else "TOTAL" if name == "Ticker" else ""
                    for name in pivot_df.index.names
                )
                blocks.append(pd.concat([inst_block, pd.DataFrame([total_row])]))
            pivot_df = pd.concat(blocks)
        else:
            total_row = pivot_df.sum()
            total_row.name = "TOTAL"
            pivot_df = pd.concat([pivot_df, pd.DataFrame([total_row])])

    # Risk map for coloring
    risk_map = pd.DataFrame()
    if not trade_df.empty:
        risk_map = trade_df.groupby(["Institution", "Ticker"])[["Dividend Risk", "Earnings Risk"]].max()

    # ---------------- WRITE EXCEL ---------------- #
    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        dividend_df.to_excel(writer, sheet_name="Dividends", index=False)
        dividend_monthly.to_excel(writer, sheet_name="Dividend Pivot", index=False)

        if not calls_df.empty:
            calls_df.drop(columns=["Include"]).to_excel(writer, sheet_name="Covered Calls", index=False)

        if not pivot_df.empty:
            pivot_df.to_excel(writer, sheet_name="Covered Calls Pivot")

        # ---- Coloring ---- #
        if not calls_df.empty:
            ws = writer.book["Covered Calls"]
            for r in range(2, ws.max_row + 1):
                div_risk = ws[f"L{r}"].value
                earn_risk = ws[f"M{r}"].value
                risk_count = sum([bool(div_risk), bool(earn_risk)])
                if risk_count == 2:
                    fill = RED_FILL
                elif risk_count == 1:
                    fill = ORANGE_FILL
                else:
                    fill = GREEN_FILL
                for c in range(1, ws.max_column + 1):
                    ws.cell(r, c).fill = fill

            ws_p = writer.book["Covered Calls Pivot"]
            for r in range(2, ws_p.max_row):
                inst = ws_p.cell(r, 1).value
                ticker = ws_p.cell(r, 2).value
                if (inst, ticker) in risk_map.index:
                    div_risk, earn_risk = risk_map.loc[(inst, ticker)]
                    risk_count = sum([bool(div_risk), bool(earn_risk)])
                    if risk_count == 2:
                        fill = RED_FILL
                    elif risk_count == 1:
                        fill = ORANGE_FILL
                    else:
                        fill = GREEN_FILL
                    for c in range(1, ws_p.max_column + 1):
                        ws_p.cell(r, c).fill = fill

    print("Analysis complete. Results written to:", OUTPUT_FILE)


# ---------------- RUN ---------------- #
if __name__ == "__main__":
    main()


Analysis complete. Results written to: /Users/mthumu/Documents/Personal Files/Finance/stocks_output_Nihal.xlsx
